# Product Recommendation System

In this notebook, a hybrid recommendation system is developed using the Olist e-commerce dataset.

The goal of this analysis is to generate product category recommendations by combining two different recommendation approaches:

- Association Rule Learning (Market Basket Analysis)
- Item-Based Collaborative Filtering

Association Rule Learning identifies categories that are frequently purchased together within the same order, while Item-Based Collaborative Filtering recommends categories based on user similarity and review behavior.

The final output is a unified recommendation dataset that can be integrated into a BI tool such as Power BI for visualization and business insights.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymysql
from mlxtend.frequent_patterns import apriori, association_rules


pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.5f' % x)

## Data Loading

The dataset is retrieved from the analytical SQL view `v_order_fact_delivered`, which contains cleaned and integrated order-level information including:

- Order metadata
- Customer identifiers
- Revenue metrics
- Product categories
- Review information
- Delivery performance metrics

This dataset serves as the analytical base for building recommendation models.

In [2]:
## SQL Connection
conn = pymysql.connect(database = "olist", user = "root", password = "160510", local_infile=True)
cursor = conn.cursor()

In [3]:
df_all = pd.read_sql("SELECT * FROM v_order_fact_delivered", conn)
df_all.head()

C:\Users\okand\AppData\Local\Temp\ipykernel_7596\383586695.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all = pd.read_sql("SELECT * FROM v_order_fact_delivered", conn)


,order_id,customer_id,customer_unique_id,order_status,purchase_ts,approved_ts,carrier_ts,delivered_ts,est_delivery_ts,days_to_deliver,is_late,total_price,total_freight,total_revenue,item_count,total_payment,payment_count,max_installments,payment_minus_revenue,review_score,review_comment_title,review_comment_message,review_creation_date,main_category_by_item,main_category_by_revenue,category_count,order_type,all_categories,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.00000,0,58.90000,13.29000,72.19000,1,72.19000,1.00000,2.00000,0.00000,5.00000,,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,cool_stuff\r,cool_stuff\r,1,Single,cool_stuff\r,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.00000,0,239.90000,19.93000,259.83000,1,259.83000,1.00000,3.00000,0.00000,4.00000,,,2017-05-13 00:00:00,pet_shop\r,pet_shop\r,1,Single,pet_shop\r,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,8.00000,0,199.00000,17.87000,216.87000,1,216.87000,1.00000,5.00000,0.00000,5.00000,,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,furniture_decor\r,furniture_decor\r,1,Single,furniture_decor\r,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.00000,0,12.99000,12.79000,25.78000,1,25.78000,1.00000,2.00000,0.00000,4.00000,,,2018-08-15 00:00:00,perfumery\r,perfumery\r,1,Single,perfumery\r,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.00000,0,199.90000,18.14000,218.04000,1,218.04000,1.00000,3.00000,0.00000,5.00000,,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,garden_tools\r,garden_tools\r,1,Single,garden_tools\r,varzea paulista,SP


## Data Preprocessing

Before building recommendation models, several preprocessing steps are applied:

### Date Conversion
Timestamp columns are converted to datetime format to ensure consistent time-based operations.

### Text Cleaning
Categorical columns are cleaned to remove unwanted characters such as:

- line breaks
- tab characters
- trailing spaces

This step ensures consistent category names and prevents data leakage during grouping operations.

### Missing Value Inspection
Null values are analyzed to identify potential issues in customer reviews, delivery timestamps, and category information.

In [4]:
# Editing the data type of the date columns:
date_cols = ["purchase_ts","approved_ts","carrier_ts","delivered_ts","review_creation_date"]
for c in date_cols:
    if c in df_all.columns:
        df_all[c] = pd.to_datetime(df_all[c], errors="coerce")

In [5]:
cat_cols = ["main_category_by_item","main_category_by_revenue","order_type","all_categories"]
for c in cat_cols:
    if c in df_all.columns:
        df_all[c] = df_all[c].astype(str).str.replace(r"[\r\n\t]", "", regex=True).str.strip()

In [6]:
df_all.isnull().sum()

order_id                       0
customer_id                    0
customer_unique_id             0
order_status                   0
purchase_ts                    0
approved_ts                   14
carrier_ts                     2
delivered_ts                   8
est_delivery_ts                0
days_to_deliver                8
is_late                        0
total_price                    0
total_freight                  0
total_revenue                  0
item_count                     0
total_payment                  1
payment_count                  1
max_installments               1
payment_minus_revenue          1
review_score                1107
review_comment_title        1107
review_comment_message      1107
review_creation_date        1108
main_category_by_item          0
main_category_by_revenue       0
category_count                 0
order_type                     0
all_categories                 0
customer_city                  0
customer_state                 0
dtype: int

# Association Rule Learning (Market Basket Analysis)

Association Rule Learning is used to identify relationships between product categories that are frequently purchased together.

This approach is widely used in e-commerce to understand purchasing behavior and support cross-selling strategies.

For example:

If customers frequently purchase **Bed Bath Table** products together with **Home Comfort** products, the system can recommend one category when the other is purchased.

To build association rules, the following steps are performed:

1. Convert the dataset into a customer-category transaction matrix
2. Apply the **Apriori algorithm** to identify frequent itemsets
3. Generate association rules using **lift** as the primary evaluation metric

In [10]:
df_all = df_all.copy()

df_all["all_categories"] = (
    df_all["all_categories"]
    .astype(str)
    .str.replace(r"\r|\n", "", regex=True)
    .str.strip()
    .str.split(",")
)

In [11]:
df_all = df_all.explode("all_categories")

In [12]:
df_all["all_categories"] = df_all["all_categories"].str.strip()

In [13]:
df_all = df_all[
    df_all["all_categories"].notna() &
    (df_all["all_categories"] != "") &
    (df_all["all_categories"] != "None") &
    (df_all["all_categories"] != "nan")
]

In [14]:
def create_customer_product_df(dataframe):
    return (
        dataframe.groupby(["customer_unique_id", "all_categories"])
        .size()
        .unstack(fill_value=0)
        .map(lambda x: 1 if x > 0 else 0)
    )

In [15]:
df_arl = create_customer_product_df(df_all)
df_arl.head()

all_categories,agro_industry_and_commerce,air_conditioning,art,arts_and_craftmanship,audio,auto,baby,bed_bath_table,books_general_interest,books_imported,books_technical,cds_dvds_musicals,christmas_supplies,cine_photo,computers,computers_accessories,consoles_games,construction_tools_construction,construction_tools_lights,construction_tools_safety,cool_stuff,costruction_tools_garden,costruction_tools_tools,diapers_and_hygiene,drinks,dvds_blu_ray,electronics,fashio_female_clothing,fashion_bags_accessories,fashion_childrens_clothes,fashion_male_clothing,fashion_shoes,fashion_sport,fashion_underwear_beach,fixed_telephony,flowers,food,food_drink,furniture_bedroom,furniture_decor,furniture_living_room,furniture_mattress_and_upholstery,garden_tools,health_beauty,home_appliances,home_appliances_2,home_comfort_2,home_confort,home_construction,housewares,industry_commerce_and_business,kitchen_dining_laundry_garden_furniture,la_cuisine,luggage_accessories,market_place,music,musical_instruments,office_furniture,party_supplies,perfumery,pet_shop,security_and_services,signaling_and_security,small_appliances,small_appliances_home_oven_and_coffee,sports_leisure,stationery,tablets_printing_image,telephony,toys,watches_gifts
customer_unique_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0000b849f77a49e4a4ce2b2a4ca5be3f,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0000f46a3911fa3c0805444483337064,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
0000f6ccb0745a6a4b88665a16c9f078,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
0004aac84e0df4da2b147fca70cf8255,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


## Apriori Algorithm

The Apriori algorithm is applied to identify frequently co-occurring product categories.

The algorithm searches for item combinations that meet a minimum support threshold.

Key parameters used:

- **min_support = 0.0001**
- **use_colnames = True**

The output of this step is a list of frequent itemsets along with their support values.

In [17]:
frequent_itemsets = apriori(
    df_arl,
    min_support=0.0001,
    use_colnames=True
)

C:\Users\okand\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [18]:
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets["length"].value_counts().sort_index()

length
1    69
2    65
Name: count, dtype: int64

### Generating Association Rules

After identifying frequent itemsets, association rules are generated using the following metrics:

- **Support** → Frequency of the item combination
- **Confidence** → Probability of purchasing the consequent given the antecedent
- **Lift** → Strength of the relationship between categories

Lift is particularly important because it measures how much more likely two categories are purchased together compared to random chance.

Rules are sorted by:

1. Lift
2. Confidence
3. Support

In [19]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

In [20]:
rules.sort_values(["lift", "confidence", "support"], ascending=False).head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(home_confort),(bed_bath_table),0.00425,0.09783,0.00059,0.13811,1.41172,1.00000,0.00017,1.04673,0.29289,0.00578,0.04465,0.07205
1,(bed_bath_table),(home_confort),0.09783,0.00425,0.00059,0.00599,1.41172,1.00000,0.00017,1.00176,0.32327,0.00578,0.00176,0.07205
2,(construction_tools_lights),(furniture_decor),0.00263,0.06709,0.00020,0.07438,1.10859,1.00000,0.00002,1.00787,0.09821,0.00281,0.00781,0.03865
3,(furniture_decor),(construction_tools_lights),0.06709,0.00263,0.00020,0.00291,1.10859,1.00000,0.00002,1.00029,0.10500,0.00281,0.00029,0.03865


## Association Rules Interpretation

Association Rule Learning was applied to identify co-purchase patterns between product categories.  
The results reveal several meaningful relationships between product categories based on support, confidence, and lift metrics.

### Key Rules

#### 1. Home Comfort → Bed & Bath Table

- **Antecedent:** home_confort  
- **Consequent:** bed_bath_table  
- **Support:** 0.00059  
- **Confidence:** 0.138  
- **Lift:** 1.41  

Interpretation:

Customers who purchase **home comfort products** are **1.41 times more likely** to also purchase **bed and bath table products** compared to random purchasing behavior.

Although the overall support is relatively low due to the sparse structure of the dataset, the lift value greater than 1 indicates a **positive association between these categories**.

This suggests a **home-related product cluster**, where customers purchasing comfort items often complement them with home textile products.

---

#### 2. Bed & Bath Table → Home Comfort

- **Confidence:** 0.006  
- **Lift:** 1.41  

Interpretation:

While the reverse relationship also exists, the confidence is significantly lower.  
This indicates that although customers buying **home comfort items frequently buy bed and bath products**, the opposite behavior is less common.

In other words:

> Customers purchasing bed and bath products do not necessarily buy home comfort products.

---

#### 3. Construction Tools Lights → Furniture Decor

- **Support:** 0.00020  
- **Confidence:** 0.074  
- **Lift:** 1.11  

Interpretation:

Customers purchasing **construction lighting tools** show a slightly higher probability of purchasing **furniture decoration products**.

The lift value above 1 suggests a **weak positive relationship**, indicating that customers performing home improvement activities may also purchase decorative items.

---

### Overall Insights

The association rule analysis highlights **home-related product relationships** within the dataset.

Several product clusters emerge:

**Home & Decoration Cluster**
- home_confort → bed_bath_table
- construction_tools_lights → furniture_decor

These patterns suggest that customers often purchase products related to **home improvement, comfort, and decoration together**.

---

### Business Implications

These insights can be used for several e-commerce strategies:

- **Cross-selling recommendations**
- **Product bundling strategies**
- **Personalized recommendation systems**
- **Homepage recommendation widgets**

For example:

Customers purchasing **home comfort products** could be recommended **bed and bath textile products** to increase basket size and improve cross-selling performance.

---

### Important Note

Due to the sparse nature of order-level product combinations in the dataset, support values are relatively low.  
However, lift values greater than 1 still reveal meaningful associations between product categories.

In [38]:
rules = rules.sort_values(["lift", "confidence", "support"], ascending=False)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(home_confort),(bed_bath_table),0.00425,0.09783,0.00059,0.13811,1.41172,1.00000,0.00017,1.04673,0.29289,0.00578,0.04465,0.07205
1,(bed_bath_table),(home_confort),0.09783,0.00425,0.00059,0.00599,1.41172,1.00000,0.00017,1.00176,0.32327,0.00578,0.00176,0.07205
2,(construction_tools_lights),(furniture_decor),0.00263,0.06709,0.00020,0.07438,1.10859,1.00000,0.00002,1.00787,0.09821,0.00281,0.00781,0.03865
3,(furniture_decor),(construction_tools_lights),0.06709,0.00263,0.00020,0.00291,1.10859,1.00000,0.00002,1.00029,0.10500,0.00281,0.00029,0.03865


In [40]:
rules_pb = rules.copy()

rules_pb["antecedent"] = rules_pb["antecedents"].apply(lambda x: ", ".join(list(x)))
rules_pb["consequent"] = rules_pb["consequents"].apply(lambda x: ", ".join(list(x)))

rules_pb["rule_label"] = rules_pb["antecedent"] + " → " + rules_pb["consequent"]

rules_pb = rules_pb[
    [
        "antecedent",
        "consequent",
        "rule_label",
        "support",
        "confidence",
        "lift",
        "leverage",
        "conviction",
        "zhangs_metric",
        "jaccard",
        "certainty",
        "kulczynski"
    ]
].copy()

rules_pb = rules_pb.sort_values(["lift", "confidence"], ascending=False).reset_index(drop=True)
rules_pb["recommendation_rank"] = rules_pb.groupby("antecedent").cumcount() + 1

rules_pb.head()

,antecedent,consequent,rule_label,support,confidence,lift,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,recommendation_rank
0,home_confort,bed_bath_table,home_confort → bed_bath_table,0.00059,0.13811,1.41172,0.00017,1.04673,0.29289,0.00578,0.04465,0.07205,1
1,bed_bath_table,home_confort,bed_bath_table → home_confort,0.00059,0.00599,1.41172,0.00017,1.00176,0.32327,0.00578,0.00176,0.07205,1
2,construction_tools_lights,furniture_decor,construction_tools_lights → furniture_decor,0.00020,0.07438,1.10859,0.00002,1.00787,0.09821,0.00281,0.00781,0.03865,1
3,furniture_decor,construction_tools_lights,furniture_decor → construction_tools_lights,0.00020,0.00291,1.10859,0.00002,1.00029,0.10500,0.00281,0.00029,0.03865,1


# Item-Based Collaborative Filtering

Item-Based Collaborative Filtering recommends categories based on user behavior patterns.

The idea behind this approach is:

> If users who liked category A also liked category B, then category B can be recommended when category A is purchased.

In this implementation:

- Customers represent rows
- Product categories represent columns
- Review scores represent interaction strength

A customer-category matrix is created and correlation between categories is computed.

In [43]:
df_all.head()

,order_id,customer_id,customer_unique_id,order_status,purchase_ts,approved_ts,carrier_ts,delivered_ts,est_delivery_ts,days_to_deliver,is_late,total_price,total_freight,total_revenue,item_count,total_payment,payment_count,max_installments,payment_minus_revenue,review_score,review_comment_title,review_comment_message,review_creation_date,main_category_by_item,main_category_by_revenue,category_count,order_type,all_categories,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.00000,0,58.90000,13.29000,72.19000,1,72.19000,1.00000,2.00000,0.00000,5.00000,,"Perfeito, produto entregue antes do combinado.",2017-09-21,cool_stuff,cool_stuff,1,Single,cool_stuff,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.00000,0,239.90000,19.93000,259.83000,1,259.83000,1.00000,3.00000,0.00000,4.00000,,,2017-05-13,pet_shop,pet_shop,1,Single,pet_shop,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,8.00000,0,199.00000,17.87000,216.87000,1,216.87000,1.00000,5.00000,0.00000,5.00000,,Chegou antes do prazo previsto e o produto sur...,2018-01-23,furniture_decor,furniture_decor,1,Single,furniture_decor,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.00000,0,12.99000,12.79000,25.78000,1,25.78000,1.00000,2.00000,0.00000,4.00000,,,2018-08-15,perfumery,perfumery,1,Single,perfumery,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.00000,0,199.90000,18.14000,218.04000,1,218.04000,1.00000,3.00000,0.00000,5.00000,,Gostei pois veio no prazo determinado .,2017-03-02,garden_tools,garden_tools,1,Single,garden_tools,varzea paulista,SP


In [45]:
df_all.isnull().sum()

order_id                       0
customer_id                    0
customer_unique_id             0
order_status                   0
purchase_ts                    0
approved_ts                   13
carrier_ts                     2
delivered_ts                   8
est_delivery_ts                0
days_to_deliver                8
is_late                        0
total_price                    0
total_freight                  0
total_revenue                  0
item_count                     0
total_payment                  1
payment_count                  1
max_installments               1
payment_minus_revenue          1
review_score                1108
review_comment_title        1108
review_comment_message      1108
review_creation_date        1109
main_category_by_item          0
main_category_by_revenue       0
category_count                 0
order_type                     0
all_categories                 0
customer_city                  0
customer_state                 0
dtype: int

In [47]:
df_all.dropna(
    subset=["review_score", "review_comment_title", "review_comment_message"],
    inplace=True
)

df_all.isnull().sum()

order_id                     0
customer_id                  0
customer_unique_id           0
order_status                 0
purchase_ts                  0
approved_ts                 13
carrier_ts                   2
delivered_ts                 8
est_delivery_ts              0
days_to_deliver              8
is_late                      0
total_price                  0
total_freight                0
total_revenue                0
item_count                   0
total_payment                1
payment_count                1
max_installments             1
payment_minus_revenue        1
review_score                 0
review_comment_title         0
review_comment_message       0
review_creation_date         1
main_category_by_item        0
main_category_by_revenue     0
category_count               0
order_type                   0
all_categories               0
customer_city                0
customer_state               0
dtype: int64

In [49]:
df_all["main_category_by_item"].value_counts().head()

main_category_by_item
bed_bath_table           9181
health_beauty            8611
sports_leisure           7430
computers_accessories    6447
furniture_decor          6170
Name: count, dtype: int64

In [51]:
df_all["main_category_by_item"].value_counts().describe()

count     72.00000
mean    1316.05556
std     2188.89481
min        2.00000
25%       73.50000
50%      239.50000
75%     1340.00000
max     9181.00000
Name: count, dtype: float64

In [53]:
categories_counts = pd.DataFrame(df_all["main_category_by_item"].value_counts())

# Removing Rare Categories

Rare categories can introduce noise and unstable correlations.

To improve recommendation quality:

- Categories with fewer than **100 observations** are removed.

This ensures that correlations are calculated on sufficiently large samples and prevents unreliable recommendations.

In [55]:
rare_categories = categories_counts[categories_counts["count"] <= 100].index

common_categories = df_all[~df_all["main_category_by_item"].isin(rare_categories)]
common_categories.shape

(93909, 30)

## Category Similarity Calculation

A correlation matrix is generated between product categories.

This matrix captures the relationship between categories based on user review behavior.

The recommendation process works as follows:

1. Select a purchased category
2. Identify categories with the highest correlation
3. Recommend the most similar category above a correlation threshold

In [57]:
common_categories["main_category_by_item"].nunique()

51

In [59]:
df_all["main_category_by_item"].nunique()

72

In [61]:
user_category_df = common_categories.pivot_table(index = ["customer_unique_id"], columns = ["main_category_by_item"], values = "review_score")

In [63]:
user_category_df.shape

(90652, 51)

Creating Item-Based Category Suggestions

In [66]:
category_name = "home_confort"
category_name = user_category_df[category_name]

In [68]:
user_category_df.corrwith(category_name).sort_values(ascending=False).head(20)

C:\Users\okand\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
C:\Users\okand\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\okand\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


main_category_by_item
home_confort                       1.00000
furniture_decor                    0.97073
bed_bath_table                    -0.11936
agro_industry_and_commerce             NaN
air_conditioning                       NaN
art                                    NaN
audio                                  NaN
auto                                   NaN
baby                                   NaN
books_general_interest                 NaN
books_technical                        NaN
christmas_supplies                     NaN
computers                              NaN
computers_accessories                  NaN
consoles_games                         NaN
construction_tools_construction        NaN
construction_tools_lights              NaN
construction_tools_safety              NaN
cool_stuff                             NaN
costruction_tools_garden               NaN
dtype: float64

## Generating Item-Based Recommendations

For each purchased category, the system identifies the most correlated category.

A minimum correlation threshold of **0.30** is applied to ensure meaningful recommendations.

The output includes:

- Recommended category
- Correlation strength

These recommendations capture behavioral similarity across product categories.

In [70]:
user_category_df.head()

main_category_by_item,agro_industry_and_commerce,air_conditioning,art,audio,auto,baby,bed_bath_table,books_general_interest,books_technical,christmas_supplies,computers,computers_accessories,consoles_games,construction_tools_construction,construction_tools_lights,construction_tools_safety,cool_stuff,costruction_tools_garden,drinks,electronics,fashion_bags_accessories,fashion_male_clothing,fashion_shoes,fashion_underwear_beach,fixed_telephony,food,food_drink,furniture_decor,furniture_living_room,garden_tools,health_beauty,home_appliances,home_appliances_2,home_confort,home_construction,housewares,industry_commerce_and_business,kitchen_dining_laundry_garden_furniture,luggage_accessories,market_place,musical_instruments,office_furniture,perfumery,pet_shop,signaling_and_security,small_appliances,sports_leisure,stationery,telephony,toys,watches_gifts
customer_unique_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,NaN,NaN,NaN,NaN,NaN,NaN,5.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0000b849f77a49e4a4ce2b2a4ca5be3f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0000f46a3911fa3c0805444483337064,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.00000,NaN,NaN,NaN
0000f6ccb0745a6a4b88665a16c9f078,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.00000,NaN,NaN
0004aac84e0df4da2b147fca70cf8255,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.00000,NaN,NaN


In [72]:
common_categories.head()

,order_id,customer_id,customer_unique_id,order_status,purchase_ts,approved_ts,carrier_ts,delivered_ts,est_delivery_ts,days_to_deliver,is_late,total_price,total_freight,total_revenue,item_count,total_payment,payment_count,max_installments,payment_minus_revenue,review_score,review_comment_title,review_comment_message,review_creation_date,main_category_by_item,main_category_by_revenue,category_count,order_type,all_categories,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.00000,0,58.90000,13.29000,72.19000,1,72.19000,1.00000,2.00000,0.00000,5.00000,,"Perfeito, produto entregue antes do combinado.",2017-09-21,cool_stuff,cool_stuff,1,Single,cool_stuff,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.00000,0,239.90000,19.93000,259.83000,1,259.83000,1.00000,3.00000,0.00000,4.00000,,,2017-05-13,pet_shop,pet_shop,1,Single,pet_shop,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,8.00000,0,199.00000,17.87000,216.87000,1,216.87000,1.00000,5.00000,0.00000,5.00000,,Chegou antes do prazo previsto e o produto sur...,2018-01-23,furniture_decor,furniture_decor,1,Single,furniture_decor,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.00000,0,12.99000,12.79000,25.78000,1,25.78000,1.00000,2.00000,0.00000,4.00000,,,2018-08-15,perfumery,perfumery,1,Single,perfumery,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.00000,0,199.90000,18.14000,218.04000,1,218.04000,1.00000,3.00000,0.00000,5.00000,,Gostei pois veio no prazo determinado .,2017-03-02,garden_tools,garden_tools,1,Single,garden_tools,varzea paulista,SP


In [74]:
corr_matrix = user_category_df.corr()

In [98]:
def item_based_cat(x, min_corr=0.30):
    correlations = corr_matrix[x].sort_values(ascending=False)

    correlations = correlations[correlations.index != x]
    correlations = correlations[correlations < 1]
    correlations = correlations[correlations >= min_corr]

    if correlations.empty:
        return None, None

    else:

        category = correlations.index[0]
        correlation = correlations.iloc[0]

    return category, correlation

In [100]:
common_categories[["item_based_suggestion", "correlation"]] = (
    common_categories["main_category_by_item"]
    .apply(item_based_cat)
    .apply(pd.Series)
)

C:\Users\okand\AppData\Local\Temp\ipykernel_7596\3390316814.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  common_categories[["item_based_suggestion", "correlation"]] = (


In [104]:
common_categories.head()

,order_id,customer_id,customer_unique_id,order_status,purchase_ts,approved_ts,carrier_ts,delivered_ts,est_delivery_ts,days_to_deliver,is_late,total_price,total_freight,total_revenue,item_count,total_payment,payment_count,max_installments,payment_minus_revenue,review_score,review_comment_title,review_comment_message,review_creation_date,main_category_by_item,main_category_by_revenue,category_count,order_type,all_categories,customer_city,customer_state,item_based_suggestion,correlation
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.00000,0,58.90000,13.29000,72.19000,1,72.19000,1.00000,2.00000,0.00000,5.00000,,"Perfeito, produto entregue antes do combinado.",2017-09-21,cool_stuff,cool_stuff,1,Single,cool_stuff,campos dos goytacazes,RJ,furniture_decor,0.96875
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.00000,0,239.90000,19.93000,259.83000,1,259.83000,1.00000,3.00000,0.00000,4.00000,,,2017-05-13,pet_shop,pet_shop,1,Single,pet_shop,santa fe do sul,SP,furniture_decor,0.87831
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,8.00000,0,199.00000,17.87000,216.87000,1,216.87000,1.00000,5.00000,0.00000,5.00000,,Chegou antes do prazo previsto e o produto sur...,2018-01-23,furniture_decor,furniture_decor,1,Single,furniture_decor,para de minas,MG,home_confort,0.97073
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.00000,0,12.99000,12.79000,25.78000,1,25.78000,1.00000,2.00000,0.00000,4.00000,,,2018-08-15,perfumery,perfumery,1,Single,perfumery,atibaia,SP,fashion_bags_accessories,0.86860
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.00000,0,199.90000,18.14000,218.04000,1,218.04000,1.00000,3.00000,0.00000,5.00000,,Gostei pois veio no prazo determinado .,2017-03-02,garden_tools,garden_tools,1,Single,garden_tools,varzea paulista,SP,computers_accessories,0.60053


# Hybrid Recommendation System

To improve recommendation quality, both methods are combined:

1. Association Rule Learning
2. Item-Based Collaborative Filtering

The final recommendation dataset merges both outputs using the purchased category as the key.

This hybrid approach provides:

- behavioral similarity recommendations
- basket-based recommendations

In [108]:
rules_pb

,antecedent,consequent,rule_label,support,confidence,lift,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,recommendation_rank
0,home_confort,bed_bath_table,home_confort → bed_bath_table,0.00059,0.13811,1.41172,0.00017,1.04673,0.29289,0.00578,0.04465,0.07205,1
1,bed_bath_table,home_confort,bed_bath_table → home_confort,0.00059,0.00599,1.41172,0.00017,1.00176,0.32327,0.00578,0.00176,0.07205,1
2,construction_tools_lights,furniture_decor,construction_tools_lights → furniture_decor,0.00020,0.07438,1.10859,0.00002,1.00787,0.09821,0.00281,0.00781,0.03865,1
3,furniture_decor,construction_tools_lights,furniture_decor → construction_tools_lights,0.00020,0.00291,1.10859,0.00002,1.00029,0.10500,0.00281,0.00029,0.03865,1


In [110]:
df_recommendation = common_categories.merge(
    rules_pb,
    left_on="main_category_by_item",
    right_on="antecedent",
    how="left"
)

In [112]:
df_recommendation = df_recommendation[
    [
        "order_id",
        "customer_unique_id",
        "main_category_by_item",
        "item_based_suggestion",
        "correlation",
        "consequent",
        "support",
        "confidence",
        "lift"
    ]
]

In [114]:
df_recommendation = df_recommendation.rename(columns={
    "main_category_by_item": "purchased_category",
    "consequent": "association_recommendation"
})

In [116]:
df_recommendation.head()

,order_id,customer_unique_id,purchased_category,item_based_suggestion,correlation,association_recommendation,support,confidence,lift
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,cool_stuff,furniture_decor,0.96875,NaN,NaN,NaN,NaN
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,pet_shop,furniture_decor,0.87831,NaN,NaN,NaN,NaN
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,furniture_decor,home_confort,0.97073,construction_tools_lights,0.00020,0.00291,1.10859
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,perfumery,fashion_bags_accessories,0.86860,NaN,NaN,NaN,NaN
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,garden_tools,computers_accessories,0.60053,NaN,NaN,NaN,NaN


In [118]:
df_recommendation.to_csv("category_recommendations.csv", index=False)

## Final Recommendation Dataset

The final dataset includes the following fields:

| Column | Description |
|------|------|
| order_id | Order identifier |
| customer_unique_id | Unique customer ID |
| purchased_category | Category purchased in the order |
| item_based_suggestion | Category recommended using collaborative filtering |
| correlation | Similarity score between categories |
| association_recommendation | Category recommended via association rules |
| support | Frequency of rule occurrence |
| confidence | Conditional probability of rule |
| lift | Strength of association |

This dataset can be exported and integrated into BI dashboards for visualization and business decision support.

## Functionalizing the Recommendation Pipeline

To make the recommendation system reusable and scalable, the entire workflow is encapsulated in a function.

This function performs:

- data preprocessing
- association rule generation
- collaborative filtering
- merging recommendation outputs

Encapsulating the logic into a function improves:

- code modularity
- reproducibility
- deployment readiness

In [127]:
## SQL Connection
conn = pymysql.connect(database = "olist", user = "root", password = "160510", local_infile=True)
cursor = conn.cursor()

In [129]:
df_all = pd.read_sql("SELECT * FROM v_order_fact_delivered", conn)
df_all.head()

C:\Users\okand\AppData\Local\Temp\ipykernel_7596\383586695.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all = pd.read_sql("SELECT * FROM v_order_fact_delivered", conn)


,order_id,customer_id,customer_unique_id,order_status,purchase_ts,approved_ts,carrier_ts,delivered_ts,est_delivery_ts,days_to_deliver,is_late,total_price,total_freight,total_revenue,item_count,total_payment,payment_count,max_installments,payment_minus_revenue,review_score,review_comment_title,review_comment_message,review_creation_date,main_category_by_item,main_category_by_revenue,category_count,order_type,all_categories,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.00000,0,58.90000,13.29000,72.19000,1,72.19000,1.00000,2.00000,0.00000,5.00000,,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,cool_stuff\r,cool_stuff\r,1,Single,cool_stuff\r,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.00000,0,239.90000,19.93000,259.83000,1,259.83000,1.00000,3.00000,0.00000,4.00000,,,2017-05-13 00:00:00,pet_shop\r,pet_shop\r,1,Single,pet_shop\r,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,8.00000,0,199.00000,17.87000,216.87000,1,216.87000,1.00000,5.00000,0.00000,5.00000,,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,furniture_decor\r,furniture_decor\r,1,Single,furniture_decor\r,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.00000,0,12.99000,12.79000,25.78000,1,25.78000,1.00000,2.00000,0.00000,4.00000,,,2018-08-15 00:00:00,perfumery\r,perfumery\r,1,Single,perfumery\r,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.00000,0,199.90000,18.14000,218.04000,1,218.04000,1.00000,3.00000,0.00000,5.00000,,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,garden_tools\r,garden_tools\r,1,Single,garden_tools\r,varzea paulista,SP


In [131]:
def create_customer_product_df(dataframe):
    return (
        dataframe.groupby(["customer_unique_id", "all_categories"])
        .size()
        .unstack(fill_value=0)
        .map(lambda x: 1 if x > 0 else 0)
    )

In [141]:
def recommendation(dataframe):
    # Editing the data type of the date columns:
    date_cols = ["purchase_ts","approved_ts","carrier_ts","delivered_ts","review_creation_date"]
    for c in date_cols:
        if c in dataframe.columns:
            dataframe[c] = pd.to_datetime(dataframe[c], errors="coerce")
            
    cat_cols = ["main_category_by_item","main_category_by_revenue","order_type","all_categories"]
    for c in cat_cols:
        if c in dataframe.columns:
            dataframe[c] = dataframe[c].astype(str).str.replace(r"[\r\n\t]", "", regex=True).str.strip()

    # --- Assocation Rule Learning

    df_copy = dataframe.copy()
    
    df_copy["all_categories"] = (
        df_copy["all_categories"]
        .astype(str)
        .str.replace(r"\r|\n", "", regex=True)
        .str.strip()
        .str.split(",")
    )

    df_copy = df_copy.explode("all_categories")

    df_copy["all_categories"] = df_copy["all_categories"].str.strip()

    df_copy = df_copy[
        df_copy["all_categories"].notna() &
        (df_copy["all_categories"] != "") &
        (df_copy["all_categories"] != "None") &
        (df_copy["all_categories"] != "nan")
    ]

    df_arl = create_customer_product_df(df_copy)

    # --- Establishment of Association Rules ---
    frequent_itemsets = apriori(
        df_arl,
        min_support=0.0001,
        use_colnames=True
    )

    rules = association_rules(
        frequent_itemsets,
        metric="lift",
        min_threshold=1.0
    )

    rules = rules.sort_values(["lift", "confidence", "support"], ascending=False)

    rules_pb = rules.copy()
    
    rules_pb["antecedent"] = rules_pb["antecedents"].apply(lambda x: ", ".join(list(x)))
    rules_pb["consequent"] = rules_pb["consequents"].apply(lambda x: ", ".join(list(x)))
    
    rules_pb["rule_label"] = rules_pb["antecedent"] + " → " + rules_pb["consequent"]
    
    rules_pb = rules_pb[
        [
            "antecedent",
            "consequent",
            "rule_label",
            "support",
            "confidence",
            "lift",
            "leverage",
            "conviction",
            "zhangs_metric",
            "jaccard",
            "certainty",
            "kulczynski"
        ]
    ].copy()
    
    rules_pb = rules_pb.sort_values(["lift", "confidence"], ascending=False).reset_index(drop=True)
    rules_pb["recommendation_rank"] = rules_pb.groupby("antecedent").cumcount() + 1

    # --- Item Based Collaborative Filtering ---

    df_copy2 = dataframe.copy()
    
    df_copy2.dropna(
        subset=["review_score", "review_comment_title", "review_comment_message"],
        inplace=True
    )

    categories_counts = pd.DataFrame(df_copy2["main_category_by_item"].value_counts())
    rare_categories = categories_counts[categories_counts["count"] <= 100].index

    common_categories = df_copy2[~df_copy2["main_category_by_item"].isin(rare_categories)]

    user_category_df = common_categories.pivot_table(index = ["customer_unique_id"], 
                                                     columns = ["main_category_by_item"], 
                                                     values = "review_score")

    corr_matrix = user_category_df.corr()

    def item_based_cat(x, min_corr=0.30):
        correlations = corr_matrix[x].sort_values(ascending=False)
    
        correlations = correlations[correlations.index != x]
        correlations = correlations[correlations < 1]
        correlations = correlations[correlations >= min_corr]
    
        if correlations.empty:
            return None, None
    
        else:
    
            category = correlations.index[0]
            correlation = correlations.iloc[0]
    
        return category, correlation

    common_categories[["item_based_suggestion", "correlation"]] = (
        common_categories["main_category_by_item"]
        .apply(item_based_cat)
        .apply(pd.Series)
    )

    df_recommendation = common_categories.merge(
        rules_pb,
        left_on="main_category_by_item",
        right_on="antecedent",
        how="left"
    )

    df_recommendation = df_recommendation[
        [
            "order_id",
            "customer_unique_id",
            "main_category_by_item",
            "item_based_suggestion",
            "correlation",
            "consequent",
            "support",
            "confidence",
            "lift"
        ]
    ]

    df_recommendation = df_recommendation.rename(columns={
        "main_category_by_item": "purchased_category",
        "consequent": "association_recommendation"
    })

    return df_recommendation
    

In [143]:
df_recommendation = recommendation(df_all)
df_recommendation.head()

C:\Users\okand\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
C:\Users\okand\AppData\Local\Temp\ipykernel_7596\4155762331.py:117: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  common_categories[["item_based_suggestion", "correlation"]] = (
C:\Users\okand\AppData\Local\Temp\ipykernel_7596\4155762331.py:117: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-doc

,order_id,customer_unique_id,purchased_category,item_based_suggestion,correlation,association_recommendation,support,confidence,lift
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,cool_stuff,furniture_decor,0.96875,NaN,NaN,NaN,NaN
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,pet_shop,furniture_decor,0.87831,NaN,NaN,NaN,NaN
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,furniture_decor,home_confort,0.97073,construction_tools_lights,0.00020,0.00291,1.10859
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,perfumery,fashion_bags_accessories,0.86860,NaN,NaN,NaN,NaN
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,garden_tools,computers_accessories,0.60053,NaN,NaN,NaN,NaN


In [145]:
df_recommendation.to_csv("category_recommendations.csv", index=False)

## Database Upload & Integration

The final recommendation dataset is exported and uploaded to the database.

A table named **recommendation_results** is created using SQL.

This enables seamless integration with:

- Power BI dashboards
- analytics pipelines
- real-time recommendation systems

In [149]:
q_create_table_rec = """
CREATE TABLE IF NOT EXISTS recommendation_results (

    order_id VARCHAR(50),
    customer_unique_id VARCHAR(50),

    purchased_category VARCHAR(100),
    item_based_suggestion VARCHAR(100),

    correlation DECIMAL(10,5),

    association_recommendation VARCHAR(100),

    support DECIMAL(12,8),
    confidence DECIMAL(12,8),
    lift DECIMAL(10,5)

);
"""

cursor.execute(q_create_table_rec)
conn.commit()

In [151]:
q_load_data_rec ="""
LOAD DATA LOCAL INFILE 'C:/Users/okand/Desktop/ecommerce/category_recommendations.csv'
INTO TABLE recommendation_results
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 ROWS;
"""

cursor.execute(q_load_data_rec)
conn.commit()

In [153]:
pd.read_sql("SELECT * FROM recommendation_results LIMIT 10;", conn)

C:\Users\okand\AppData\Local\Temp\ipykernel_7596\2581235604.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT * FROM recommendation_results LIMIT 10;", conn)


,order_id,customer_unique_id,purchased_category,item_based_suggestion,correlation,association_recommendation,support,confidence,lift
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,cool_stuff,furniture_decor,0.96875,,0.00000,0.00000,0.00000
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,pet_shop,furniture_decor,0.87831,,0.00000,0.00000,0.00000
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,furniture_decor,home_confort,0.97073,construction_tools_lights,0.00020,0.00291,1.10859
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,perfumery,fashion_bags_accessories,0.86860,,0.00000,0.00000,0.00000
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,garden_tools,computers_accessories,0.60053,,0.00000,0.00000,0.00000
5,00048cc3ae777c65dbb7d2a0634bc1ea,85c835d128beae5b4ce8602c491bf385,housewares,bed_bath_table,0.63512,,0.00000,0.00000,0.00000
6,00054e8431b9d7675808bcb819fb4a32,635d9ac1680f03288e72ada3a1035803,telephony,toys,0.96077,,0.00000,0.00000,0.00000
7,000576fe39319847cbb9d288c5617fa6,fda4476abb6307ab3c415b7e6d026526,garden_tools,computers_accessories,0.60053,,0.00000,0.00000,0.00000
8,0005a1a1728c9d785b8e2b08b904576c,639d23421f5517f69d0c3d6e6564cf0e,health_beauty,drinks,0.96850,,0.00000,0.00000,0.00000
9,0005f50442cb953dcd1d21e1fb923495,0782c41380992a5a533489063df0eef6,books_technical,,0.00000,,0.00000,0.00000,0.00000


In [157]:
conn.close()

# Conclusion

In this notebook, a hybrid recommendation system was developed using the Olist e-commerce dataset.

Two complementary recommendation techniques were implemented:

- Association Rule Learning
- Item-Based Collaborative Filtering

By combining these approaches, the system captures both:

- basket-level purchase behavior
- category similarity based on user feedback

The final output is a structured recommendation dataset that can be directly integrated into business intelligence tools for visualization and decision-making.